In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_dna_member.job_manager as job_manager
import lib_dna_member.generate_population as gp
import lib_dna_member.member_features as features
import lib_dna_member.utils as utils
from lib_dna_member.s3 import member_dna_input_data_validator
from databricks.feature_engineering import FeatureEngineeringClient

In [0]:
%run ../../config/utils

In [0]:
def generate_member(job):
    """
    Generate the features associated with member

    Parameters:
        job (managers.JobManager): object which manages the Spark App
    Returns:
        (pyspark.sql.DataFrame): dna with new features
    """
    dna = job.tables["population"]
    orig_cols = dna.columns

    dna = features.feature_member_master(job, dna)
    dna = features.feature_member_extended(job, dna)
    dna = features.feature_team_member_ind(job, dna)
    dna = features.feature_trial_member_ind(job, dna)

    # dna = utils.cache_df(dna)
    dna = features.feature_base_mfi(job, dna)
    dna = features.feature_member_history(job, dna)
    dna = features.feature_calculated_mfi(job, dna)
    dna = features.feature_distance(job, dna)
    dna = features.feature_tenure(job, dna)

    # dna = utils.cache_df(dna)

    dna = features.feature_days_until_exp(job, dna)
    dna = features.feature_days_since_last_rnwl(job, dna)
    dna = features.feature_num_of_rnwls(job, dna)
    dna = features.feature_quotient_id(job, dna)

    # dna = dna.persist()
    # dna.count()

    # dna = utils.cache_df(dna)

    dna = dna.drop(
        *[
            col
            for col in orig_cols
            if col not in ["MBRSHP_SID", "FISCAL_WEEK_END"]
        ]
    )

    return dna

In [0]:
job = job_manager.JobManager(spark, intermediate_all_tables_dict, member_dna_config_path)

In [0]:
recency_lookback_duration = job.config["params"].get( 
    "recency_lookback_duration", {}
)
member_dna_input_data_validator(
    silver_skeleton, silver_master_member_extended, silver_master_member, silver_master_member_history, silver_master_census_tract, silver_quotient_id,
    recency_lookback_duration=recency_lookback_duration,
    spark=spark
)



In [0]:
job.read_table("skeleton")
job.read_table("member")
job.read_table("member_extended")
job.read_table("member_history")
job.read_table("census_tract")
job.read_table("quotient_id")

In [0]:
population = gp.generate_population(job)
job.tables["population"] = population

features = generate_member(job)
features = features.withColumn('MBRSHP_SID', f.coalesce('MBRSHP_SID', f.lit(-1))) # TODO REVISAR

### Save results

In [0]:
spark.sql(f"DELETE FROM {fs_cubes_member}")

fe = FeatureEngineeringClient()

fe.write_table(
    name=fs_cubes_member,
    df=features,
    mode="merge"
)